# Module X3: Bayesian inference

**Goal:** learn the **Bayesian** way of updating beliefs with data, compute posteriors **by hand** (conjugate formulas, a grid, and a Metropolis sampler written from scratch), and be able to say clearly when and why to use it, and how it differs from the frequentist methods in Modules 8 and 8b.

**You will learn:** priors, likelihoods and posteriors · the Beta–Binomial, Normal–Normal and Gamma–Poisson models · credible intervals · the effect of the prior and the sample size · grid approximation · a Bayesian A/B test ·
Markov chain Monte Carlo (random-walk Metropolis) and its diagnostics (acceptance rate, $\hat R$, effective sample size) · posterior predictive checks · reporting and prior-sensitivity analysis

**Maths in this module:** Bayes' rule for parameters $p(\theta\mid y)\propto p(y\mid\theta)\,p(\theta)$ · Beta, Gamma and Normal conjugacy · credible intervals · Monte Carlo estimation of probabilities · Metropolis acceptance ratio · convergence diagnostics

> 🔗 **Prerequisites:** Modules 7 (Bayes' theorem, distributions) and 7b (likelihood). This module needs no extra packages (NumPy and SciPy only); for bigger models see "Where next" at the end.

In [ ]:
# Setup: find the course data folder (works locally and in Google Colab)
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    import subprocess
    if not Path("/content/data-analysis-learnbook").exists():
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/PathornKiti/data-analysis-learnbook.git",
                        "/content/data-analysis-learnbook"], check=True)
    DATA = Path("/content/data-analysis-learnbook/data")
else:
    DATA = next(p / "data" for p in [Path.cwd(), *Path.cwd().parents]
                if (p / "data" / "students.csv").exists())
print("Data folder:", DATA)

In [ ]:
import math

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style="whitegrid", palette="colorblind")
trial = pd.read_csv(DATA / "app_trial.csv")
students = pd.read_csv(DATA / "students.csv")
desk = pd.read_csv(DATA / "service_desk.csv")

## 1. The idea

**Frequentist** methods ask: *if the true value were θ, how surprising is my data?* **Bayesian** methods ask the question people usually want answered: *given my data, what do I now believe about θ?*

$$\underbrace{p(\theta\mid y)}_{\text{posterior}}\;\propto\;\underbrace{p(y\mid\theta)}_{\text{likelihood}}\;\times\;\underbrace{p(\theta)}_{\text{prior}}$$

* The **prior** encodes what you knew or assumed *before* the data (from earlier studies, physical limits, or deliberately vague).
* The **likelihood** is how well each θ explains the data (the same likelihood as in Module 7b).
* The **posterior** is the updated belief, a full probability distribution over θ. Everything you want (a best estimate, an interval, the probability that θ exceeds some value) is read from it.

A **95% credible interval** $[a,b]$ means: *given the data and prior, there's a 95% probability that θ lies in $[a,b]$.* (A confidence interval doesn't mean that: it describes the long-run behaviour of the procedure.) This direct interpretation is the main appeal.

## 2. The Beta–Binomial model, by hand

Take the conversion data: `control` converted **17 of 100** users. Let $p$ be the true conversion rate. The likelihood for $x$ successes in $n$ trials is binomial. If the prior is **Beta($\alpha,\beta$)** (a flexible distribution on $[0,1]$ with mean $\alpha/(\alpha+\beta)$), the posterior is again a Beta, a property called **conjugacy**:

$$p\sim\text{Beta}(\alpha,\beta),\ \ x\mid p\sim\text{Bin}(n,p)\ \Longrightarrow\ p\mid x\sim\text{Beta}(\alpha+x,\ \beta+n-x)$$

Think of $\alpha$ and $\beta$ as **prior pseudo-counts** of successes and failures. Beta(1,1) is flat ("every rate equally likely a priori").

In [ ]:
conv = {g: (int(trial.loc[trial["group"] == g, "converted"].sum()), 100) for g in ("control", "tutorial", "gamified")}

def beta_posterior(x, n, a0=1, b0=1):
    return a0 + x, b0 + n - x

rows = []
for g, (x, n) in conv.items():
    a, b = beta_posterior(x, n)
    post = stats.beta(a, b)
    lo, hi = post.interval(0.95)
    rows.append((g, f"{x}/{n}", x / n, post.mean(), lo, hi))
pd.DataFrame(rows, columns=["group", "data", "MLE (x/n)", "posterior mean", "95% CrI low", "95% CrI high"]).round(3)

In [ ]:
p = np.linspace(0, 0.6, 600)
fig, ax = plt.subplots(figsize=(7.5, 3.8))
for g, (x, n) in conv.items():
    ax.plot(p, stats.beta(*beta_posterior(x, n)).pdf(p), label=f"{g}: {x}/{n}")
ax.plot(p, stats.beta(1, 1).pdf(p), color="grey", linestyle=":", label="flat prior Beta(1,1)")
ax.set_xlabel("True conversion rate p"); ax.set_ylabel("Posterior density"); ax.legend(); plt.show()

With a flat prior the posterior mean $(x+1)/(n+2)=0.176$ is slightly above the raw 17% (it's pulled gently toward 0.5 by the two pseudo-observations), and the 95% credible interval for control is **0.109 to 0.256**, essentially the Wilson interval from Module 8b.
With a flat prior and decent data, Bayesian and frequentist intervals **agree numerically**; they differ in *meaning* and in what you can do next (see §5).

## 3. How much does the prior matter?

It depends on the **amount of data**. Compare a flat prior, a weakly informative Beta(2,8) (mean 0.2, worth 10 pseudo-observations) and a strong Beta(20,80) (mean 0.2, worth 100) with a *small* sample (2 of 10) and a *larger* one (17 of 100).

In [ ]:
priors = {"flat Beta(1,1)": (1, 1), "weak Beta(2,8)": (2, 8), "strong Beta(20,80)": (20, 80)}
rows = []
for label, (x, n) in {"2 of 10": (2, 10), "17 of 100": (17, 100)}.items():
    for pname, (a0, b0) in priors.items():
        a, b = beta_posterior(x, n, a0, b0)
        rows.append((label, pname, x / n, stats.beta(a, b).mean()))
sens = pd.DataFrame(rows, columns=["data", "prior", "raw proportion", "posterior mean"]).round(3)
print(sens.pivot(index="prior", columns="data", values="posterior mean"))

With **10 observations** the prior matters a lot (posterior means range 0.20–0.25); with **100** the data dominate and the posteriors nearly coincide. This is why Bayesians do a **prior sensitivity analysis**: report how much the conclusion changes under reasonable alternative priors.
Priors are *not* a flaw to hide: they make assumptions explicit. Frequentist methods make assumptions too (about models, stopping rules, etc.), just less visibly.

## 4. Grid approximation: Bayes without formulas

You don't need conjugacy. Put a **grid** of candidate values for $p$, compute $\text{prior}\times\text{likelihood}$ at each, and **normalise** so the values sum to 1. It's slow in many dimensions but perfect for learning and for one-parameter problems with *any* prior shape.

In [ ]:
def grid_posterior(x, n, prior_density, n_grid=2000):
    grid = np.linspace(0.0005, 0.9995, n_grid)
    unnormalised = prior_density(grid) * stats.binom.pmf(x, n, grid)
    return grid, unnormalised / unnormalised.sum()

# 1. flat prior: should reproduce the conjugate answer exactly
grid, post_flat = grid_posterior(17, 100, lambda p: np.ones_like(p))
exact = stats.beta(18, 84).pdf(grid); exact /= exact.sum()
print(f"flat prior: grid mean {np.sum(grid * post_flat):.4f} vs exact {18 / 102:.4f}; largest pointwise difference {np.abs(post_flat - exact).max():.1e}")

# 2. a prior with no conjugate form: 'I believe the rate is near 0.30 (sd 0.05)', e.g. from last year's campaign
prior_near_30 = lambda p: np.exp(-0.5 * ((p - 0.30) / 0.05) ** 2)
grid, post_informed = grid_posterior(17, 100, prior_near_30)
cdf = np.cumsum(post_informed)
lo, hi = grid[np.searchsorted(cdf, 0.025)], grid[np.searchsorted(cdf, 0.975)]
print(f"informed prior near 0.30: posterior mean {np.sum(grid * post_informed):.3f}, 95% CrI ({lo:.3f}, {hi:.3f})")

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 3.6))
ax.plot(grid, prior_near_30(grid) / prior_near_30(grid).sum(), color="grey", linestyle=":", label="prior: near 0.30")
ax.plot(grid, stats.binom.pmf(17, 100, grid) / stats.binom.pmf(17, 100, grid).sum(), color="tab:green", label="likelihood (17/100), rescaled")
ax.plot(grid, post_informed, color="tab:red", linewidth=2, label="posterior")
ax.set_xlim(0, 0.5); ax.set_xlabel("p"); ax.set_ylabel("relative plausibility"); ax.legend(); plt.show()

The posterior **compromises** between the prior (centred at 0.30) and the data (0.17): it sits in between, closer to the more informative (narrower) of the two. A prior that disagrees strongly with the data is a warning sign: re-examine the prior *and* the data.

## 5. A Bayesian A/B test

Draw many values from each group's posterior; each pair of draws is one *plausible world*. Then any question becomes a **proportion of draws**:

In [ ]:
rng = np.random.default_rng(0)
n_draws = 400_000
p_game = stats.beta(*beta_posterior(*conv["gamified"])).rvs(n_draws, random_state=rng)
p_ctrl = stats.beta(*beta_posterior(*conv["control"])).rvs(n_draws, random_state=rng)
diff = p_game - p_ctrl

print(f"P(gamified converts better than control) = {np.mean(diff > 0):.4f}")
print(f"posterior mean difference = {diff.mean():.3f}, 95% credible interval ({np.quantile(diff, 0.025):.3f}, {np.quantile(diff, 0.975):.3f})")
print(f"P(improvement of more than 5 points)     = {np.mean(diff > 0.05):.3f}")
print(f"P(difference within +/-2 points: 'practically equivalent') = {np.mean(np.abs(diff) < 0.02):.4f}")
odds_ratio = (p_game / (1 - p_game)) / (p_ctrl / (1 - p_ctrl))
print(f"odds ratio: median {np.median(odds_ratio):.2f}, 95% CrI ({np.quantile(odds_ratio, 0.025):.2f}, {np.quantile(odds_ratio, 0.975):.2f})")

xs = np.linspace(0, 1, 20001)
f = stats.beta(*beta_posterior(*conv["gamified"])).pdf(xs) * stats.beta(*beta_posterior(*conv["control"])).cdf(xs)
exact_prob = ((f[:-1] + f[1:]) / 2 * np.diff(xs)).sum()                       # trapezoid rule: P(p_game > p_ctrl) = integral of f_game * F_ctrl
print(f"exact P(gamified > control) by numerical integration = {exact_prob:.4f}  (Monte Carlo error is small)")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.hist(diff, bins=80, density=True, alpha=0.6)
ax.axvline(0, color="k"); ax.axvspan(-0.02, 0.02, color="grey", alpha=0.25, label="'practically no difference' region (ROPE)")
lo, hi = np.quantile(diff, [0.025, 0.975]); ax.axvline(lo, color="tab:red", linestyle="--"); ax.axvline(hi, color="tab:red", linestyle="--", label="95% credible interval")
ax.set_xlabel("Gamified minus control conversion rate"); ax.legend(fontsize=8); plt.show()

Compare with the frequentist result from Module 8b (difference 0.16, 95% CI 0.04 to 0.28, $p=0.009$). The numbers are close, but the Bayesian statements are the ones decision-makers want:
**"There's a 99.5% probability gamified converts better, and a 96% probability it improves conversion by more than 5 points."** The posterior also supports decisions with costs (expected loss) and is straightforward to extend.
A **ROPE** (region of practical equivalence) makes "no meaningful difference" a probability statement (here only about 1% mass is within ±2 points), the Bayesian counterpart of the equivalence test.

## 6. Continuous data: the Normal model

For a mean $\mu$ with known $\sigma$ and a $N(\mu_0,\tau^2)$ prior, the posterior is normal, with **precisions** (inverse variances) adding:

$$\frac1{\tau_n^2}=\frac1{\tau^2}+\frac{n}{\sigma^2},\qquad \mu_n=\tau_n^2\Big(\frac{\mu_0}{\tau^2}+\frac{n\bar x}{\sigma^2}\Big)$$

The posterior mean is a **precision-weighted average** of the prior mean and the sample mean. With an uninformative prior and unknown $\sigma$, the posterior for $\mu$ is exactly the $t$ distribution behind the usual confidence interval.

In [ ]:
x = students["exam_score"].to_numpy()
n, xbar, s = len(x), x.mean(), x.std(ddof=1)

flat = stats.t(n - 1, xbar, s / math.sqrt(n))
print(f"flat prior, unknown sigma: posterior of the mean is t with centre {xbar:.2f}, scale {s / math.sqrt(n):.3f}")
print(f"   95% credible interval ({flat.interval(0.95)[0]:.2f}, {flat.interval(0.95)[1]:.2f})  = the t confidence interval")
print(f"   P(true mean > 65) = {1 - flat.cdf(65):.4f};   P(true mean > 68) = {1 - flat.cdf(68):.3f}")

def normal_posterior(mu0, tau, xbar, sigma, n):
    precision = 1 / tau ** 2 + n / sigma ** 2
    return (mu0 / tau ** 2 + n * xbar / sigma ** 2) / precision, precision ** -0.5

mn, sn = normal_posterior(65, 3, xbar, s, n)
print(f"\ninformative prior N(65, 3^2) (sigma taken as {s:.1f}): posterior N({mn:.2f}, {sn:.3f}); 95% CrI ({mn - 1.96 * sn:.2f}, {mn + 1.96 * sn:.2f})")

A sceptical prior (a mean of 65 with a standard deviation of 3) pulls the posterior mean from 67.94 only to **67.77**: with 200 observations the data swamp the prior. Posterior probabilities answer questions the p-value can't: the mean is above 65 with probability > 99.99%, but only about 47% likely to exceed 68.

## 7. Counts: the Gamma–Poisson model

For a rate $\lambda$ with Poisson counts $y_1..y_n$ and a **Gamma($a,b$)** prior (shape $a$, rate $b$; mean $a/b$), the posterior is Gamma($a+\sum y_i,\ b+n$). Tickets per hour (`service_desk.csv`), with a prior worth about "four tickets per hour, with the strength of one hour's data":

In [ ]:
tickets = desk["tickets"].to_numpy()
a0, b0 = 2, 0.5                                                  # prior mean a0/b0 = 4 per hour
a_post, b_post = a0 + tickets.sum(), b0 + len(tickets)
post = stats.gamma(a_post, scale=1 / b_post)
lo, hi = post.interval(0.95)
print(f"{len(tickets)} hours, {tickets.sum()} tickets (raw mean {tickets.mean():.4f})")
print(f"posterior Gamma({a_post:.0f}, rate {b_post:.1f}): mean {post.mean():.4f}, 95% CrI ({lo:.3f}, {hi:.3f}); P(rate > 4) = {1 - post.cdf(4):.5f}")

## 8. When there is no formula: MCMC (a Metropolis sampler from scratch)

Most realistic models (logistic regression, hierarchical models) have no closed-form posterior. **Markov chain Monte Carlo** draws samples from a posterior known only up to a constant. The **random-walk Metropolis** algorithm:

1. Start at some $\theta^{(0)}$.
2. Propose $\theta^*=\theta^{(t)}+\text{noise}$ (a normal step of chosen size).
3. Compute the **acceptance ratio** $r=\dfrac{p(y\mid\theta^*)\,p(\theta^*)}{p(y\mid\theta^{(t)})\,p(\theta^{(t)})}$ (the constant cancels!).
4. Accept with probability $\min(1,r)$ (so uphill moves are always accepted, downhill sometimes); otherwise stay.

The chain wanders over the posterior, spending time in proportion to its probability. Here: the exam scores with **unknown mean $\mu$ and standard deviation $\sigma$**, priors $\mu\sim N(65,20^2)$ and $\ln\sigma\sim N(\ln10,1^2)$ (weakly informative).

In [ ]:
def log_posterior(mu, log_sigma, data, mu0=65, mu_sd=20, ls0=math.log(10), ls_sd=1.0):
    log_likelihood = -len(data) * log_sigma - ((data - mu) ** 2).sum() / (2 * math.exp(2 * log_sigma))
    return log_likelihood + stats.norm.logpdf(mu, mu0, mu_sd) + stats.norm.logpdf(log_sigma, ls0, ls_sd)

def metropolis(data, n_iter, start, step, rng):
    current = np.array(start, dtype=float)
    current_lp = log_posterior(*current, data)
    samples, accepted = np.empty((n_iter, 2)), 0
    for i in range(n_iter):
        proposal = current + rng.normal(0, step, size=2)
        proposal_lp = log_posterior(*proposal, data)
        if math.log(rng.random()) < proposal_lp - current_lp:          # accept with probability min(1, ratio), on the log scale
            current, current_lp, accepted = proposal, proposal_lp, accepted + 1
        samples[i] = current
    return samples, accepted / n_iter

rng = np.random.default_rng(1)
starts = ([50, math.log(5)], [80, math.log(20)], [60, math.log(8)], [75, math.log(15)])   # four chains from very different starting points
chains, rates = [], []
for start in starts:
    draws, rate = metropolis(x, 6000, start, step=[0.75, 0.07], rng=rng)
    chains.append(draws[1000:]); rates.append(rate)                                      # discard the first 1,000 draws (burn-in)
chains = np.array(chains)
print("acceptance rates:", np.round(rates, 2))

In [ ]:
mu_draws = chains[:, :, 0]
fig, axes = plt.subplots(1, 2, figsize=(12, 3.4))
for c in range(4):
    axes[0].plot(mu_draws[c], linewidth=0.5, alpha=0.8)
axes[0].set_title("trace plots of mu (4 chains): a 'hairy caterpillar' means good mixing"); axes[0].set_xlabel("iteration"); axes[0].set_ylabel("mu")
axes[1].hist(mu_draws.ravel(), bins=50, density=True, alpha=0.6, label="MCMC")
grid_mu = np.linspace(65, 71, 300); axes[1].plot(grid_mu, flat.pdf(grid_mu), color="tab:red", label="analytic t posterior")
axes[1].legend(); axes[1].set_xlabel("mu")
plt.tight_layout(); plt.show()

### Diagnostics: did the sampler work?

* **Acceptance rate**: for random-walk Metropolis aim for roughly 20–50%. Near 100% = steps too tiny (slow exploration); near 0% = too large.
* **$\hat R$ (Gelman–Rubin)**: with $m$ chains of length $L$, compare within-chain variance $W$ with between-chain variance $B=L\,\operatorname{Var}(\text{chain means})$: $\hat R=\sqrt{\dfrac{\frac{L-1}{L}W+\frac1LB}{W}}$. It should be **very close to 1** (below about 1.01).
* **Effective sample size (ESS)**: successive draws are autocorrelated, so $N$ draws are worth only $N/(1+2\sum\rho_k)$ independent ones.

In [ ]:
def r_hat(chains_1d):
    m, L = chains_1d.shape
    W = chains_1d.var(axis=1, ddof=1).mean()
    B = L * chains_1d.mean(axis=1).var(ddof=1)
    return math.sqrt(((L - 1) / L * W + B / L) / W)

def effective_sample_size(chain):
    c = chain - chain.mean()
    acf = np.correlate(c, c, "full")[len(c) - 1:] / (c @ c)
    total = 1.0
    for k in range(1, len(acf)):
        if acf[k] < 0.05:
            break
        total += 2 * acf[k]
    return len(chain) / total

print(f"R-hat for mu: {r_hat(mu_draws):.3f}")
print("effective sample sizes per chain (of 5,000 draws):", [int(effective_sample_size(c)) for c in mu_draws])
lo, hi = np.quantile(mu_draws, [0.025, 0.975])
print(f"MCMC posterior of mu: mean {mu_draws.mean():.2f}, 95% CrI ({lo:.2f}, {hi:.2f})   |   analytic: mean {xbar:.2f}, CrI ({flat.interval(0.95)[0]:.2f}, {flat.interval(0.95)[1]:.2f})")
print(f"posterior mean of sigma: {np.exp(chains[:, :, 1]).mean():.2f}   (sample sd {s:.2f})")

The four chains started far apart (means 50–80) but converged to the same answer, with $\hat R\approx1.003$; each chain's 5,000 draws are worth only about 350–460 independent samples (the effective sample size is much smaller than the number of draws), which is still plenty for a mean and an interval. The MCMC interval (66.45 to 69.40) matches the analytic one (66.47 to 69.40).
For real work, use mature samplers (Hamiltonian Monte Carlo in **Stan** or **PyMC**) that handle many parameters, and always check $\hat R$, ESS and trace plots.

## 9. Checking the model: posterior predictive checks

A posterior is only as good as the model. **Simulate replicated datasets from the fitted model** (drawing parameters from the posterior, then data from the likelihood) and compare features of the replicates with the real data. If the data look unlike the replicates, the model is missing something.

In [ ]:
rng = np.random.default_rng(2)
n_rep = 4000
sigma2 = (n - 1) * s ** 2 / rng.chisquare(n - 1, n_rep)                    # posterior draws of sigma^2 (flat prior, normal model)
mu_rep = rng.normal(xbar, np.sqrt(sigma2 / n))                             # then mu given sigma^2
replicated = rng.normal(mu_rep[:, None], np.sqrt(sigma2)[:, None], size=(n_rep, n))

checks = {"maximum": lambda v: v.max(axis=-1), "minimum": lambda v: v.min(axis=-1),
          "standard deviation": lambda v: v.std(axis=-1), "skewness": lambda v: stats.skew(v, axis=-1)}
rows = []
for name, f in checks.items():
    observed, rep = float(f(x)), f(replicated)
    rows.append((name, observed, rep.mean(), np.mean(rep >= observed)))
print(pd.DataFrame(rows, columns=["statistic", "observed", "mean of replicates", "P(replicate >= observed)"]).round(3).to_string(index=False))

A "posterior predictive p-value" near 0 or 1 flags a problem. The normal model reproduces the **spread, minimum and maximum** well, but **fails the skewness check** (observed 0.38 vs about 0 in replicates; $P\approx0.01$): exam scores are slightly right-skewed, which a symmetric normal model cannot produce.
That is harmless for *inference about the mean* with $n=200$ (the central limit theorem protects us), but it would matter for *predicting an individual student's score*. A good report says exactly this.

## 10. Using Bayesian results in a thesis

**When Bayesian methods are especially useful:** small samples with genuine prior information; hierarchical/multilevel structure; decisions that need probabilities of hypotheses or expected losses; sequential updating as data arrive; complex models with constraints.

**Cautions:**
* Choose and **justify priors**; state them. Prefer *weakly informative* priors over "flat" ones for scale parameters.
* Do a **prior sensitivity analysis** (rerun with 2–3 reasonable alternatives).
* Report **diagnostics** ($\hat R$, ESS, trace plots) and **posterior predictive checks**.
* Don't use posterior probabilities of hypotheses as if they were p-values: they depend on the prior. **Bayes factors** compare how well two models predict the data but are very sensitive to priors; use with care.
* Say "credible interval", not "confidence interval", and interpret it as a probability statement *conditional on the model and prior*.

### Reporting template
> *We estimated each group's conversion rate with a Beta–Binomial model and a flat Beta(1, 1) prior (results were insensitive to a Beta(2, 8) prior). The posterior mean difference (gamified − control) was 0.157 (95% credible interval 0.039 to 0.274; 100 users per group), with a 99.5% posterior probability that the gamified rate is higher and a 96% probability that the improvement exceeds 5 percentage points.
> Exam-score means were modelled with a normal likelihood; posterior inference by random-walk Metropolis (4 chains × 5,000 post-burn-in draws; R̂ = 1.003; ESS > 340 per chain) agreed with the analytic posterior. A posterior predictive check showed the normal model reproduces the spread but not the slight right skew of the scores.*

## Where next
For hierarchical and non-conjugate models use **PyMC** or **Stan** (via `cmdstanpy`); **ArviZ** provides diagnostics and plots. Hierarchical Bayesian versions of Module X1's mixed models and of regression (Module 9) are natural next steps.

## Recap

* **Posterior ∝ likelihood × prior.** Conjugate pairs (Beta–Binomial, Normal–Normal, Gamma–Poisson) give formulas; a **grid** works for one parameter; **MCMC** handles the rest.
* Credible intervals have the direct probability interpretation; with flat priors and enough data they match frequentist intervals numerically.
* The prior matters for small data (do a **sensitivity analysis**) and hardly at all for large data.
* Check MCMC ($\hat R$, ESS, traces) and the model (**posterior predictive checks**).

**Next:** `exercises.ipynb`, [`math-notes.md`](math-notes.md), then the **project checkpoint** in the README.